# For PyTorch Users

> **NOTE:**
> **This doesn't contain a model that you can readily use, and is not intended for beginner users.** 
> That said, if you'd like to build a deep learning model with PyTorch, we've provided utilities and a general scaffold you can use to get started.

In [21]:
import sys
import os

# Don't remove this. It tells Python where to find the utilities we've written
BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
sys.path.append(BASE_DIR)

In [22]:
# Imports

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

In [23]:
from userkits.torch_dataset import MinecraftTorchDataset
from userkits.features import *
from userkits.utils import *

from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Make sure all images are 224x224
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])



train_dataset = MinecraftTorchDataset("../train_data", transform=transform)

print(train_dataset[0][2])



Found 1483 images across 29 biomes
3365_3508_5415.png


In [11]:


from torchvision import models
from torchvision.models import ResNet18_Weights

# Use the updated weights argument instead of pretrained
weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Freeze early layers
for name, param in model.named_parameters():
    if any(layer in name for layer in ["conv1", "layer1", "layer2"]):
        param.requires_grad = False

# Detect number of biomes
data_path = "../train_data"
num_biomes = len([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])
print("Detected number of biomes:", num_biomes)

# Replace final layer
model.fc = torch.nn.Linear(model.fc.in_features, num_biomes)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


Detected number of biomes: 29


# Training Loop

In [24]:

num_epochs = 50      # Maximum epochs
batch_size = 32
patience = 5         # Stop if no improvement after this many epochs
delta = 1e-4         # Minimum change in val loss to qualify as improvement

# Split dataset
train_dataset, val_dataset, _ = train_dataset.split_dataset(train_ratio=0.8, val_ratio=0.2, test_ratio=0.0)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=False)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

best_val_loss = float('inf')
epochs_no_improve = 0

# Precompute validation labels on device
val_labels_all = torch.tensor([val_dataset.biome_to_class[l] for l in val_dataset.labels], device=device)

for epoch in range(num_epochs):
    torch.mps.empty_cache()
    # --- Training ---
    model.train()
    running_train_loss = 0.0
    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = torch.tensor([train_dataset.biome_to_class[l] for l in labels], device=device)

        optimizer.zero_grad()
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    running_val_loss = 0.0
    start_idx = 0
    with torch.no_grad():
        for images, _, _ in val_loader:
            images = images.to(device)
            batch_size_val = images.size(0)
            labels = val_labels_all[start_idx:start_idx + batch_size_val]
            start_idx += batch_size_val

            logits = model(images)
            val_loss = F.cross_entropy(logits, labels)
            running_val_loss += val_loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- Early Stopping Check ---
    if avg_val_loss < best_val_loss - delta:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"Validation improved. Model saved.")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s).")
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

print("Training complete. Best validation loss:", best_val_loss)



Epoch 1/50 | Train Loss: 2.5212 | Val Loss: 1.5225
Validation improved. Model saved.
Epoch 2/50 | Train Loss: 1.0640 | Val Loss: 0.9327
Validation improved. Model saved.
Epoch 3/50 | Train Loss: 0.5594 | Val Loss: 0.7057
Validation improved. Model saved.
Epoch 4/50 | Train Loss: 0.3412 | Val Loss: 0.5894
Validation improved. Model saved.
Epoch 5/50 | Train Loss: 0.1983 | Val Loss: 0.5318
Validation improved. Model saved.
Epoch 6/50 | Train Loss: 0.1237 | Val Loss: 0.4885
Validation improved. Model saved.
Epoch 7/50 | Train Loss: 0.0789 | Val Loss: 0.4639
Validation improved. Model saved.
Epoch 8/50 | Train Loss: 0.0586 | Val Loss: 0.4375
Validation improved. Model saved.
Epoch 9/50 | Train Loss: 0.0455 | Val Loss: 0.4135
Validation improved. Model saved.
Epoch 10/50 | Train Loss: 0.0340 | Val Loss: 0.4271
No improvement for 1 epoch(s).
Epoch 11/50 | Train Loss: 0.0263 | Val Loss: 0.4045
Validation improved. Model saved.
Epoch 12/50 | Train Loss: 0.0220 | Val Loss: 0.4040
No improvement

# Evaluation

In [25]:

# --- Load best model ---
model.load_state_dict(torch.load("best_model.pth"))
model.eval()  # set to evaluation mode

# --- Evaluation dataset ---
eval_dataset = MinecraftTorchDataset(
    "/Users/ethannguyen-huu/Projects/blockography-ai/eval_data", 
    mode='eval',
    transform=transform
)
eval_dataloader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

# Map class indices from training to evaluation dataset
eval_dataset.class_to_biome = train_dataset.class_to_biome

predictions = []
file_ids = []

# --- Run inference ---
with torch.no_grad():
    for images, _, batch_file_ids in eval_dataloader:
        images = images.to(device)
        logits = model(images)
        _, predicted_indices = torch.max(logits, dim=1)

        for idx, file_id in zip(predicted_indices, batch_file_ids):
            predicted_biome = eval_dataset.class_to_biome[idx.item()]
            predictions.append(predicted_biome)
            file_ids.append(file_id)

# --- Save predictions ---
save_predictions(predictions, file_ids, output_file='../output/ridge_predictions.csv')

print(f"Predictions saved for {len(predictions)} images.")



Found 1486 eval images
Saved ../output/ridge_predictions.csv
Predictions saved for 1486 images.
